In [1]:
import json
from pathlib import Path
from collections import Counter, defaultdict
import yaml

In [2]:
cwd = Path.cwd()
CONFIG_PATH = cwd.parent / "config" / "config.yaml"

with open(CONFIG_PATH, "r") as f:
    cfg = yaml.safe_load(f)

BASE_DIR = Path(cfg["paths"]["htem_raw_data_root"]).resolve()
assert BASE_DIR.exists(), f"Filtered path does not exist: {BASE_DIR}"

SAMPLES_SUBDIR = "samples"
XRD_KEY = "xrd_angle"

In [3]:
def load_json(path: Path):
    try:
        with path.open("r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None

In [4]:
def get_all_sample_files(base_dir: Path):
    sample_files = []

    for lib_dir in base_dir.iterdir():
        if not lib_dir.is_dir():
            continue

        samples_dir = lib_dir / SAMPLES_SUBDIR
        if not samples_dir.exists():
            continue

        for sample_file in samples_dir.glob("sample *.json"):
            sample_files.append(sample_file)

    return sample_files

In [5]:
def extract_xrd_signature(sample_json):
    if XRD_KEY not in sample_json:
        return None, "missing_xrd_angle"

    xrd = sample_json[XRD_KEY]

    if not isinstance(xrd, list):
        return None, "not_array"

    try:
        signature = tuple(float(v) for v in xrd)
    except Exception:
        return None, "non_numeric"

    return signature, None

In [6]:
def scan_xrd_angles(sample_files):
    signatures = {}
    failures = defaultdict(list)

    for sample_path in sample_files:
        sample = load_json(sample_path)
        if sample is None:
            failures["unreadable"].append(sample_path)
            continue

        signature, error = extract_xrd_signature(sample)

        if error:
            failures[error].append(sample_path)
            continue

        signatures[sample_path] = signature

    return signatures, failures

In [7]:
def determine_canonical_signature(signatures):
    counter = Counter(signatures.values())
    canonical_signature, count = counter.most_common(1)[0]
    return canonical_signature

In [8]:
def find_mismatches(signatures, canonical_signature):
    mismatches = []

    for path, sig in signatures.items():
        if sig != canonical_signature:
            mismatches.append(path)

    return mismatches

In [9]:
sample_files = get_all_sample_files(BASE_DIR)
print(f"Total sample files found: {len(sample_files)}")

signatures, failures = scan_xrd_angles(sample_files)

length_counts = Counter(len(sig) for sig in signatures.values())

print("\nXRD ANGLE LENGTH DISTRIBUTION:")
for length, count in sorted(length_counts.items()):
    print(f"  Length {length}: {count} samples")

canonical_signature = determine_canonical_signature(signatures)
canonical_length = len(canonical_signature)

print(f"\nCanonical xrd_angle length: {canonical_length}")

mismatches = find_mismatches(signatures, canonical_signature)

print(f"\nSamples with mismatched ordering or values: {len(mismatches)}")

if failures:
    print("\nFailures by category:")
    for k, v in failures.items():
        print(f"  {k}: {len(v)} samples")

Total sample files found: 9856

XRD ANGLE LENGTH DISTRIBUTION:
  Length 661: 9680 samples
  Length 681: 176 samples

Canonical xrd_angle length: 661

Samples with mismatched ordering or values: 185


In [10]:
print(len(failures))

0


In [11]:
# 661 values makes sense for 19 to 52 degress stepping by 0.05
# investigate the 185 mismatched samples with 681 length or 661 values but not 19-52 by 0.05

print(canonical_signature) # confirms 661, 19-51 by 0.05

(19.0, 19.05, 19.1, 19.15, 19.2, 19.25, 19.3, 19.35, 19.4, 19.45, 19.5, 19.55, 19.6, 19.65, 19.7, 19.75, 19.8, 19.85, 19.9, 19.95, 20.0, 20.05, 20.1, 20.15, 20.2, 20.25, 20.3, 20.35, 20.4, 20.45, 20.5, 20.55, 20.6, 20.65, 20.7, 20.75, 20.8, 20.85, 20.9, 20.95, 21.0, 21.05, 21.1, 21.15, 21.2, 21.25, 21.3, 21.35, 21.4, 21.45, 21.5, 21.55, 21.6, 21.65, 21.7, 21.75, 21.8, 21.85, 21.9, 21.95, 22.0, 22.05, 22.1, 22.15, 22.2, 22.25, 22.3, 22.35, 22.4, 22.45, 22.5, 22.55, 22.6, 22.65, 22.7, 22.75, 22.8, 22.85, 22.9, 22.95, 23.0, 23.05, 23.1, 23.15, 23.2, 23.25, 23.3, 23.35, 23.4, 23.45, 23.5, 23.55, 23.6, 23.65, 23.7, 23.75, 23.8, 23.85, 23.9, 23.95, 24.0, 24.05, 24.1, 24.15, 24.2, 24.25, 24.3, 24.35, 24.4, 24.45, 24.5, 24.55, 24.6, 24.65, 24.7, 24.75, 24.8, 24.85, 24.9, 24.95, 25.0, 25.05, 25.1, 25.15, 25.2, 25.25, 25.3, 25.35, 25.4, 25.45, 25.5, 25.55, 25.6, 25.65, 25.7, 25.75, 25.8, 25.85, 25.9, 25.95, 26.0, 26.05, 26.1, 26.15, 26.2, 26.25, 26.3, 26.35, 26.4, 26.45, 26.5, 26.55, 26.6, 26.65

In [12]:
def get_mismatch_signatures(signatures, canonical_signature):
    mismatched_signatures = {}
    for key, sig in signatures.items():
        if sig != canonical_signature:
            if sig not in mismatched_signatures:
                mismatched_signatures[sig] = []
            mismatched_signatures[sig].append(key)
    return mismatched_signatures    

In [13]:
unnatural_signatures = get_mismatch_signatures(signatures, canonical_signature)
len(unnatural_signatures)

10

In [14]:
# There are 10 unnatural xrd_angle values/orderings
# There is a chance that some of them that are 661 long are rotations of the main signature

number_sigs_661 = []
number_sigs_681 = []
for sig in unnatural_signatures:
    if len(sig) == 661:
        number_sigs_661.append(sig)
    elif len(sig) == 681:
        number_sigs_681.append(sig)
print(len(number_sigs_661))
print(len(number_sigs_681))

8
2


In [15]:
# Function to see if two arrays contain the same elements as each other
def permutated_arrays(arr1, arr2):
    if type(arr1) != type(arr2):
        print("Type mismatch")
        return False
        
    if len(arr1) != len(arr2):
        print("Legth mismatch")
        return False

    for v in arr1:
        if v not in arr2: return False

    return True

In [16]:
permutated_sigs = []
for sig in number_sigs_661:
    if permutated_arrays(sig, canonical_signature):
        permutated_sigs.append(sig)
print(len(permutated_sigs))

8


In [17]:
# Ahh, so all 8 have all 661 values, just in a different order
# This means that the samples with 681 are the only ones that should be omitted
print(number_sigs_681)

[(28.0, 28.05, 28.1, 28.15, 28.2, 28.25, 28.3, 28.35, 28.4, 28.45, 28.5, 28.55, 28.6, 28.65, 28.7, 28.75, 28.8, 28.85, 28.9, 28.95, 29.0, 29.05, 29.1, 29.15, 29.2, 29.25, 29.3, 29.35, 29.4, 29.45, 29.5, 29.55, 29.6, 29.65, 29.7, 29.75, 29.8, 29.85, 29.9, 29.95, 30.0, 30.05, 30.1, 30.15, 30.2, 30.25, 30.3, 30.35, 30.4, 30.45, 30.5, 30.55, 30.6, 30.65, 30.7, 30.75, 30.8, 30.85, 30.9, 30.95, 31.0, 31.05, 31.1, 31.15, 31.2, 31.25, 31.3, 31.35, 31.4, 31.45, 31.5, 31.55, 31.6, 31.65, 31.7, 31.75, 31.8, 31.85, 31.9, 31.95, 32.0, 32.05, 32.1, 32.15, 32.2, 32.25, 32.3, 32.35, 32.4, 32.45, 32.5, 32.55, 32.6, 32.65, 32.7, 32.75, 32.8, 32.85, 32.9, 32.95, 33.0, 33.05, 33.1, 33.15, 33.2, 33.25, 33.3, 33.35, 33.4, 33.45, 33.5, 33.55, 33.6, 33.65, 33.7, 33.75, 33.8, 33.85, 33.9, 33.95, 34.0, 34.05, 34.1, 34.15, 34.2, 34.25, 34.3, 34.35, 34.4, 34.45, 34.5, 34.55, 34.6, 34.65, 34.7, 34.75, 34.8, 34.85, 34.9, 34.95, 35.0, 35.05, 35.1, 35.15, 35.2, 35.25, 35.3, 35.35, 35.4, 35.45, 35.5, 35.55, 35.6, 35.6

In [18]:
# Number of samples with these two 681 length signature
total_ommitted_samples = 0
for sig in number_sigs_681:
    total_ommitted_samples += len(unnatural_signatures[sig])
print(f"Number of omitted samples: {total_ommitted_samples}")

Number of omitted samples: 176


In [19]:
# same as the number before: valid
# conclusion: 9680 usable samples

# question: as 176 (samples) / 44 (samples/library) = 4 (libriaries)
# are there exactly 4 out of 224 libraries that were measured differently?

import re

TARGET_LEN = 681
EXPECTED_SAMPLES_PER_LIBRARY = 44

In [20]:
SAMPLE_ID_RE = re.compile(r"sample\s+(\d+)\.json$", re.IGNORECASE)

def get_sample_id_from_filename(path: Path):
    m = SAMPLE_ID_RE.search(path.name)
    return int(m.group(1)) if m else None

def get_xrd_length(sample_json):
    if sample_json is None:
        return None, "unreadable"

    if XRD_KEY not in sample_json:
        return None, "missing_xrd_angle"

    xrd = sample_json[XRD_KEY]
    if not isinstance(xrd, list):
        return None, "not_array"

    return len(xrd), None

In [21]:
def find_libraries_all_samples_target_len(
    base_dir: Path,
    target_len: int,
    expected_samples_per_library: int
):
    qualifying = {}  # lib_id -> sorted sample_ids
    per_library_summary = {}  # lib_id -> dict stats
    failures_global = defaultdict(int)

    for lib_dir in base_dir.iterdir():
        if not lib_dir.is_dir():
            continue

        lib_id = lib_dir.name
        samples_dir = lib_dir / SAMPLES_SUBDIR
        if not samples_dir.exists():
            continue

        sample_files = sorted(samples_dir.glob("sample *.json"))
        sample_count = len(sample_files)

        # Track lengths + issues for this library
        target_len_sample_ids = []
        non_target_len_sample_ids = []
        issues = defaultdict(int)

        for sf in sample_files:
            sample_json = load_json(sf)
            length, err = get_xrd_length(sample_json)

            sid = get_sample_id_from_filename(sf)
            if sid is None:
                # Fallback to "unknown"; still count the file
                sid = -1

            if err:
                issues[err] += 1
                failures_global[err] += 1
                non_target_len_sample_ids.append(sid)
                continue

            if length == target_len:
                target_len_sample_ids.append(sid)
            else:
                non_target_len_sample_ids.append(sid)

        per_library_summary[lib_id] = {
            "sample_files": sample_count,
            "target_len_count": len(target_len_sample_ids),
            "non_target_len_count": len(non_target_len_sample_ids),
            "issues": dict(issues),
        }

        # Qualifying condition: exactly expected sample files AND all are target length
        if sample_count == expected_samples_per_library and len(target_len_sample_ids) == expected_samples_per_library:
            qualifying[lib_id] = sorted(target_len_sample_ids)

    return qualifying, per_library_summary, dict(failures_global)


qualifying, per_library_summary, failures_global = find_libraries_all_samples_target_len(
    BASE_DIR,
    TARGET_LEN,
    EXPECTED_SAMPLES_PER_LIBRARY
)

print(f"Qualifying libraries (all {EXPECTED_SAMPLES_PER_LIBRARY} samples have xrd_angle length {TARGET_LEN}): {len(qualifying)}")
print("Library IDs:", sorted(qualifying.keys()))

Qualifying libraries (all 44 samples have xrd_angle length 681): 4
Library IDs: ['8335', '8336', '8337', '8338']


In [22]:
# Confirm your suspicion: 4 libraries * 44 samples = 176 samples total
total_target_samples_in_qualifying_libs = sum(len(v) for v in qualifying.values())

print(f"Total samples in qualifying libraries: {total_target_samples_in_qualifying_libs}")
print(f"Expected if 4 libraries are outliers: {4 * EXPECTED_SAMPLES_PER_LIBRARY}")

if len(qualifying) != 4:
    print("\nWARNING: Number of qualifying libraries is not 4.")

for lib_id in sorted(qualifying.keys()):
    sample_ids = qualifying[lib_id]
    print(f"\nLibrary {lib_id} -> {len(sample_ids)} samples with len {TARGET_LEN}:")
    print(sample_ids)

Total samples in qualifying libraries: 176
Expected if 4 libraries are outliers: 176

Library 8335 -> 44 samples with len 681:
[301210, 301211, 301212, 301213, 301214, 301215, 301216, 301217, 301218, 301219, 301220, 301221, 301222, 301223, 301224, 301225, 301226, 301227, 301228, 301229, 301230, 301231, 301232, 301233, 301234, 301235, 301236, 301237, 301238, 301239, 301240, 301241, 301242, 301243, 301244, 301245, 301246, 301247, 301248, 301249, 301250, 301251, 301252, 301253]

Library 8336 -> 44 samples with len 681:
[305962, 305963, 305964, 305965, 305966, 305967, 305968, 305969, 305970, 305971, 305972, 305973, 305974, 305975, 305976, 305977, 305978, 305979, 305980, 305981, 305982, 305983, 305984, 305985, 305986, 305987, 305988, 305989, 305990, 305991, 305992, 305993, 305994, 305995, 305996, 305997, 305998, 305999, 306000, 306001, 306002, 306003, 306004, 306005]

Library 8337 -> 44 samples with len 681:
[301298, 301299, 301300, 301301, 301302, 301303, 301304, 301305, 301306, 301307, 30

In [23]:
# Inquiry was correct, 4 libraries with 176 samples total were measured different (681) from the rest of the 220:
# 8335, 8336, 8337, 8338 

# Remove these 4 libraries and make separate "Filtered" directory